# 🚀 HappyGen Studio - Anima (Qwen LLM) Server
This server runs the **Anima** architecture which uses a Qwen LLM as the text encoder. This allows it to understand complex natural language prompts instead of just tags!

### Instructions:
1. Go to **Runtime -> Change runtime type -> Select T4 GPU**.
2. Run all cells below.
3. Copy the public **Cloudflare Tunnel URL** (`https://xxxx.trycloudflare.com`).
4. In HappyGen Web App, click the **Settings / Backend** icon in the header, paste the URL, and click **Test Connection**!

In [ ]:
# Cell 1: Install Dependencies
!pip uninstall -y torchaudio 2>/dev/null
!pip install -q diffusers transformers accelerate safetensors sentencepiece fastapi uvicorn pydantic pycloudflared nest_asyncio python-multipart


In [ ]:
# Cell 2: Load Anima and start the server
import os, io, base64, uuid, threading, time
import torch
import PIL.Image
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
from pycloudflared import try_cloudflare
import nest_asyncio
from diffusers import DiffusionPipeline

print("Loading Anima Base v1.0 (Qwen Encoder). This takes a few minutes...")
pipe = DiffusionPipeline.from_pretrained(
    "CalamitousFelicitousness/Anima-1.0-Base-Diffusers",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    custom_pipeline="CalamitousFelicitousness/Anima-1.0-Base-Diffusers"
).to("cuda")

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])
nest_asyncio.apply()

tasks = {}

class Txt2ImgRequest(BaseModel):
    prompt: str
    negative_prompt: str = "worst quality, low quality, score_1, score_2, score_3"
    steps: int = 20
    cfg_scale: float = 7.0
    width: int = 1024
    height: int = 1024
    seed: int = -1
    loras: list = []
    civitai_api_key: str = ""

def _encode_image(image):
    buffered = io.BytesIO()
    image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")


def manual_fuse_lora(pipe, lora_path, weight=1.0, is_unfuse=False):
    from safetensors.torch import load_file
    import torch
    
    multiplier = -float(weight) if is_unfuse else float(weight)
    
    try:
        state_dict = load_file(lora_path)
    except:
        return
        
    unet_map = {name.replace(".", "_"): (name, module) for name, module in pipe.transformer.named_modules() if hasattr(module, 'weight')}
    te_map = {}
    if hasattr(pipe, 'text_encoder'):
        te_map = {name.replace(".", "_"): (name, module) for name, module in pipe.text_encoder.named_modules() if hasattr(module, 'weight')}
        
    lora_groups = {}
    for key, tensor in state_dict.items():
        if "lora_down" in key or "lora_up" in key or "alpha" in key:
            base_key = key.split(".")[0]
            if base_key.startswith("lora_unet_"):
                mapped_name = base_key[len("lora_unet_"):]
                target_map = unet_map
            elif base_key.startswith("lora_transformer_"):
                mapped_name = base_key[len("lora_transformer_"):]
                target_map = unet_map
            elif base_key.startswith("lora_te_"):
                mapped_name = base_key[len("lora_te_"):]
                target_map = te_map
            else:
                continue
                
            if mapped_name not in target_map:
                continue
                
            real_name, module = target_map[mapped_name]
            if real_name not in lora_groups:
                lora_groups[real_name] = {'module': module}
                
            if "lora_down.weight" in key:
                lora_groups[real_name]['down'] = tensor
            elif "lora_up.weight" in key:
                lora_groups[real_name]['up'] = tensor
            elif "alpha" in key:
                lora_groups[real_name]['alpha'] = tensor.item()
                
    for real_name, group in lora_groups.items():
        if 'up' in group and 'down' in group:
            module = group['module']
            up = group['up'].to(device=module.weight.device, dtype=torch.float32)
            down = group['down'].to(device=module.weight.device, dtype=torch.float32)
            alpha = group.get('alpha', up.shape[1])
            rank = up.shape[1]
            scale = alpha / rank
            
            with torch.no_grad():
                if len(up.shape) == 2 and len(down.shape) == 2:
                    delta = (up @ down) * scale * multiplier
                    module.weight.data += delta.to(module.weight.dtype)
                elif len(up.shape) == 4 and len(down.shape) == 4:
                    up = up.squeeze()
                    down = down.squeeze()
                    if len(up.shape) == 2 and len(down.shape) == 2:
                        delta = (up @ down).unsqueeze(2).unsqueeze(3) * scale * multiplier
                        module.weight.data += delta.to(module.weight.dtype)

def _do_generate(req: Txt2ImgRequest, task_id: str):
    try:
        seed = req.seed if req.seed >= 0 else int(torch.randint(0, 2**32, (1,)).item())
        generator = torch.Generator("cuda").manual_seed(seed)
        
        # The model responds best to natural language with some quality prefix tags
        prompt_str = f"masterpiece, best quality, {req.prompt}"
        neg_prompt_str = req.negative_prompt

        import requests, os
        lora_dir = "/content/LoRAs"
        os.makedirs(lora_dir, exist_ok=True)
        if req.loras:
            print(f"Applying {len(req.loras)} LoRAs...")
            for lora in req.loras:
                name = lora.get("fileName") or lora.get("name")
                url = lora.get("downloadUrl")
                weight = lora.get("weight", 0.85)
                if not name or not url: continue
                if not name.endswith(".safetensors"): name += ".safetensors"
                path = os.path.join(lora_dir, name)
                if not os.path.exists(path):
                    print(f"Downloading LoRA {name}...")
                    headers = {"Authorization": f"Bearer {req.civitai_api_key}"} if req.civitai_api_key else {}
                    r = requests.get(url, headers=headers, stream=True)
                    if r.status_code == 200:
                        with open(path, "wb") as f:
                            for chunk in r.iter_content(chunk_size=8192):
                                f.write(chunk)
                    else:
                        print(f"Failed to download {name}: HTTP {r.status_code}")
                        continue
                try:
                    if not hasattr(pipe, "load_lora_weights"):
                try:
                    manual_fuse_lora(pipe, path, weight=weight, is_unfuse=False)
                    print(f"Successfully applied {name} (weight: {weight})")
                except Exception as e:
                    raise ValueError(f"Failed to apply LoRA {name} manually! Error: {e}")

        with torch.inference_mode():
            image = pipe(
                prompt=prompt_str,
                negative_prompt=neg_prompt_str,
                num_inference_steps=req.steps,
                guidance_scale=req.cfg_scale,
                width=req.width,
                height=req.height,
                generator=generator
            ).images[0]
            
        for lora in req.loras:
            manual_fuse_lora(pipe, os.path.join(lora_dir, lora.get("fileName") or lora.get("name") + ".safetensors"), weight=lora.get("weight", 0.85), is_unfuse=True)
        tasks[task_id] = {"status": "completed", "result": {"images": [_encode_image(image)], "source": "Anima (Qwen)"}}
    except Exception as e:
        tasks[task_id] = {"status": "failed", "error": str(e)}

@app.post("/sdapi/v1/txt2img")
def txt2img(req: Txt2ImgRequest):
    task_id = str(uuid.uuid4())
    tasks[task_id] = {"status": "processing"}
    threading.Thread(target=_do_generate, args=(req, task_id)).start()
    return {"task_id": task_id}

@app.get("/async/status/{task_id}")
def get_status(task_id: str):
    return tasks.get(task_id, {"status": "not_found"})

@app.get("/")
def health(): return {"status": "online", "base_model": "Anima-Base-v1.0 (Qwen)"}

threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"), daemon=True).start()
time.sleep(3)
import os
os.system("pkill -f cloudflared")
time.sleep(2)
try:
    tunnel = try_cloudflare(port=8000)
    print(f"\n🎉 COPY THIS URL: {tunnel.tunnel}\n")
except Exception as e:
    print("Cloudflared failed. Trying alternative tunnel (LocalTunnel)...")
    import subprocess
    os.system("npm install -g localtunnel > /dev/null 2>&1")
    p = subprocess.Popen(["lt", "--port", "8000"], stdout=subprocess.PIPE)
    url = p.stdout.readline().decode().strip().split("is: ")[1]
    print(f"\n🎉 COPY THIS URL: {url}\n")
    print("⚠️ NOTE: LocalTunnel requires you to enter the Colab IP on first visit.")
    print("Go to this URL to find your Colab IP: https://ipv4.icanhazip.com/")
